# 01 – Data Cleaning

**Project:** SocialPulse – Social Media Topic & Community Clustering

**Goal of this notebook:** turn the raw Bluesky data into a clean dataset that is ready for topic modeling and network analysis.

**Input:** `data/posts.csv`, `data/follows.csv`
**Output:** `data/posts_clean.csv`, `data/edges_clean.csv`

**Cleaning steps**
1. Load and inspect the raw data
2. Remove missing values and duplicates
3. Clean the text (links, mentions, extra spaces)
4. Remove very short posts and spam-like posts
5. Keep only follow edges between users we have text for
6. Save the clean files

## 1. Imports and loading the raw data

In [22]:
import pandas as pd
import re

posts = pd.read_csv("data/posts.csv")
edges = pd.read_csv("data/follows.csv")

print("posts:", posts.shape)
print("follow edges:", edges.shape)
posts.head()

posts: (25332, 10)
follow edges: (51828, 2)


,uri,author_did,author_handle,text,created_at,likes,reposts,replies,keyword,seed_domain
0,at://did:plc:n25ixtfn3ywdab7fgb5hhwyw/app.bsky...,did:plc:n25ixtfn3ywdab7fgb5hhwyw,directorog.bsky.social,Artificial intelligence is now cloning videos ...,2026-11-22T20:04:54.905Z,0,0,0,artificial intelligence,tech
1,at://did:plc:2hjhi62d67j7cuyy2cvljx6a/app.bsky...,did:plc:2hjhi62d67j7cuyy2cvljx6a,ladywaterood.bsky.social,substack.com/@berniesande...,2026-11-22T20:03:51.958Z,0,0,0,artificial intelligence,tech
2,at://did:plc:lqw2j2lci7buazep7uh7dnwc/app.bsky...,did:plc:lqw2j2lci7buazep7uh7dnwc,jane-lia.bsky.social,"Also, the way they distinguish between the *tr...",2026-11-22T20:03:00.756Z,1,0,0,artificial intelligence,tech
3,at://did:plc:ibig5lbg6a7pmquexoxczkft/app.bsky...,did:plc:ibig5lbg6a7pmquexoxczkft,lawrencebhatti98.bsky.social,Did you know artificial intelligence can now c...,2026-11-22T20:02:20.044Z,0,0,0,artificial intelligence,tech
4,at://did:plc:5zww7zorx2ajw7hqrhuix3ba/app.bsky...,did:plc:5zww7zorx2ajw7hqrhuix3ba,byteseu.bsky.social,AI reshapes how internships for NU students ap...,2026-11-22T20:00:34+00:00,0,0,0,artificial intelligence,tech


## 2. First look at the data

Before cleaning, we check what we have: column types, missing values, how balanced the two domains are, and how many different authors there are.

In [23]:
posts.info()
print()
print(posts["seed_domain"].value_counts())
print()
print("unique authors:", posts["author_did"].nunique())
posts[["likes", "reposts", "replies"]].describe()

<class 'pandas.DataFrame'>
RangeIndex: 25332 entries, 0 to 25331
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   uri            25332 non-null  str  
 1   author_did     25332 non-null  str  
 2   author_handle  25332 non-null  str  
 3   text           24879 non-null  str  
 4   created_at     25332 non-null  str  
 5   likes          25332 non-null  int64
 6   reposts        25332 non-null  int64
 7   replies        25332 non-null  int64
 8   keyword        25332 non-null  str  
 9   seed_domain    25332 non-null  str  
dtypes: int64(3), str(7)
memory usage: 1.9 MB

seed_domain
tech             2938
politics         2893
science          2859
entertainment    2831
climate          2799
education        2783
sports           2758
finance          2740
health           2731
Name: count, dtype: int64

unique authors: 16286


,likes,reposts,replies
count,25332.000000,25332.000000,25332.000000
mean,4.440589,0.866019,0.456142
std,37.106658,9.588212,5.296574
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,2.000000,0.000000,1.000000
max,1980.000000,644.000000,770.000000


## 3. Remove missing values and duplicates

- Drop rows with no text, author, or timestamp, since they can't be used.
- Convert `created_at` to a real datetime so we can analyze topics over time later.
- Drop duplicate posts (same post URI).

In [24]:
n_before = len(posts)

posts = posts.dropna(subset=["text", "author_did", "created_at"])
posts = posts[posts["text"].str.strip() != ""]

posts["created_at"] = pd.to_datetime(posts["created_at"], errors="coerce", utc=True)
posts = posts.dropna(subset=["created_at"])

posts = posts.drop_duplicates(subset="uri")

print(f"{n_before} -> {len(posts)} posts after basic cleaning")

25332 -> 24356 posts after basic cleaning


## 4. Clean the text

Topic models work on meaning, so we remove noise:
- URLs
- @mentions (usernames carry no topic information)
- the `#` symbol (we keep the word itself, as in #AI → AI)
- extra spaces and line breaks

We also record a few spam signals (number of hashtags and links) **before** cleaning, because cleaning would remove them. The original text is kept in `text`, and the cleaned version goes in `text_clean`.

In [25]:
def clean_text(t):
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)   # remove links
    t = re.sub(r"@[\w.\-]+", " ", t)               # remove @mentions
    t = t.replace("#", "")                         # keep hashtag word, drop symbol
    t = re.sub(r"\s+", " ", t)                     # collapse spaces/newlines
    return t.strip()

# spam signals from the ORIGINAL text
posts["n_hashtags"] = posts["text"].str.count(r"#\w+")
posts["n_links"]    = posts["text"].str.count(r"https?://")

posts["text_clean"] = posts["text"].apply(clean_text)
posts["n_words"]    = posts["text_clean"].str.split().str.len()

posts[["text", "text_clean", "n_words"]].sample(5, random_state=1)

,text,text_clean,n_words
7338,⚽ SECOND HALF — Brentford U21 2–0 Hull City U2...,⚽ SECOND HALF — Brentford U21 2–0 Hull City U2...,16
19957,NASA posted about last night's meteor. \n\nVid...,NASA posted about last night's meteor. Video a...,45
20994,It's like asking what happens if the sun disap...,It's like asking what happens if the sun disap...,54
5769,"Face it, America elected a Russian Asset as pr...","Face it, America elected a Russian Asset as pr...",46
117,"Like this? ""Huang was also notorious for his m...","Like this? ""Huang was also notorious for his m...",37


## 5. Remove low-quality posts

Rules and why:
- **Under 5 words:** too short to carry a topic ("lol", "agree").
- **More than 5 hashtags:** typical of promotional or spam posts.
- **Duplicate text:** the same message posted repeatedly (spam or bots), even when it has different post IDs.
- **More than 50 posts from one author:** stops a single heavy poster from dominating a topic.

Each filter prints how many posts it removed, so every decision is documented.

In [26]:
MIN_WORDS = 5
MAX_HASHTAGS = 5
MAX_POSTS_PER_AUTHOR = 50

def report(step, before, after):
    print(f"{step}: removed {before - after} posts ({after} left)")

n = len(posts)
posts = posts[posts["n_words"] >= MIN_WORDS]
report("short posts", n, len(posts)); n = len(posts)

posts = posts[posts["n_hashtags"] <= MAX_HASHTAGS]
report("hashtag spam", n, len(posts)); n = len(posts)

posts["text_key"] = posts["text_clean"].str.lower()
posts = posts.drop_duplicates(subset="text_key")
report("duplicate text", n, len(posts)); n = len(posts)

posts = posts.groupby("author_did").head(MAX_POSTS_PER_AUTHOR)
report("heavy posters", n, len(posts))

print("\nremaining authors:", posts["author_did"].nunique())
print(posts["seed_domain"].value_counts())

short posts: removed 1700 posts (22656 left)
hashtag spam: removed 585 posts (22071 left)
duplicate text: removed 793 posts (21278 left)
heavy posters: removed 826 posts (20452 left)

remaining authors: 14563
seed_domain
science          2457
politics         2411
finance          2321
entertainment    2304
health           2295
tech             2273
education        2259
climate          2257
sports           1875
Name: count, dtype: int64


## 6. Clean the follow network

The raw edge list has one row per "A follows B". Most follow targets are accounts we have no posts for, so they are useless for linking topics to communities.

We keep only edges where **both ends are authors in our clean post set**, and remove self-follows and duplicate edges.

In [27]:
authors = set(posts["author_did"])

edges = edges.drop_duplicates()
edges = edges[edges["source"] != edges["target"]]

edges_clean = edges[edges["source"].isin(authors) & edges["target"].isin(authors)]

print("raw edges:", len(edges))
print("usable edges:", len(edges_clean))
print("users appearing in edges:", len(set(edges_clean["source"]) | set(edges_clean["target"])))

raw edges: 51827
usable edges: 2514
users appearing in edges: 1822


## 7. Save clean data

These files are the input for the next notebook, `02_topic_modeling.ipynb`.

In [28]:
posts_out = posts.drop(columns=["text_key"])
posts_out.to_csv("data/posts_clean.csv", index=False)
edges_clean.to_csv("data/edges_clean.csv", index=False)

print("posts_clean:", posts_out.shape)
print("edges_clean:", edges_clean.shape)
print("date range:", posts_out["created_at"].min(), "->", posts_out["created_at"].max())

posts_clean: (20452, 14)
edges_clean: (2514, 2)
date range: 2020-11-20 15:23:32+00:00 -> 2026-11-22 21:02:58.903000+00:00


## Summary

- Started with **N** raw posts and **M** raw edges (fill in the numbers from the outputs above).
- Final dataset: **X** posts from **Y** authors and **Z** edges between them.
- Main filters: short posts, hashtag spam, duplicate text, heavy posters.
- **Limitation:** posts were collected by keyword search, so the dataset reflects who discusses these keywords on Bluesky, not the whole platform.